## PySpark Streaming

Voorheen hebben we gemerkt met een bestaande, volledige dataset die reeds aanwezig was in de cluster.
Echter is er vaak een nood aan data binnen te halen van verscheidene databronnen, deze om te zetten naar een bruikbaar formaat en te bewaren in een datawarehouse.
Dit is exact wat er nodig is voor het ETL principe of Extract-Transform-Load.
Een belangrijk onderdeel hiervan is de Pyspark Streaming module.
De documentatie van deze module kan [hier](https://spark.apache.org/docs/latest/streaming-programming-guide.html) gevonden worden.
Het wordcount example kan ook hierin geschreven worden, namelijk:

In [1]:
# make bucket for this notebook in MinIO
from minio import Minio
import os

client = Minio(
    endpoint="minio1:9000",
    access_key="bigdata",
    secret_key="bigdata123",
    secure=False,
    region='eu-west-1'
)

# Check verbinding
try:
    print(client.list_buckets())
    print("Verbonden met MinIO")
except Exception as e:
    print("Fout bij verbinden:", e)

bucket_name = "04-streaming"

# Stap 1: bucket aanmaken als die niet bestaat
if not client.bucket_exists(bucket_name):
    client.make_bucket(bucket_name)
    print(f"Bucket '{bucket_name}' aangemaakt")

# Stap 2: bucket leegmaken
for obj in client.list_objects(bucket_name):
    client.remove_object(bucket_name, obj.object_name)
    print(f"Verwijderd: {obj.object_name}")

[Bucket(name='01-mapreduce', creation_date=datetime.datetime(2026, 3, 3, 13, 11, 43, 563000, tzinfo=datetime.timezone.utc)), Bucket(name='02-spark', creation_date=datetime.datetime(2026, 3, 10, 13, 11, 25, 375000, tzinfo=datetime.timezone.utc)), Bucket(name='03-mllib', creation_date=datetime.datetime(2026, 3, 17, 14, 8, 33, 764000, tzinfo=datetime.timezone.utc)), Bucket(name='04-streaming', creation_date=datetime.datetime(2026, 3, 24, 15, 5, 50, 95000, tzinfo=datetime.timezone.utc)), Bucket(name='mijnclibucket', creation_date=datetime.datetime(2026, 3, 6, 13, 6, 30, 467000, tzinfo=datetime.timezone.utc)), Bucket(name='mijnpythonbucket', creation_date=datetime.datetime(2026, 2, 24, 14, 8, 57, 985000, tzinfo=datetime.timezone.utc)), Bucket(name='typb-streaming-nosql', creation_date=datetime.datetime(2026, 4, 3, 8, 4, 12, 62000, tzinfo=datetime.timezone.utc))]
Verbonden met MinIO
Verwijderd: input/


In [11]:
%%file networkwordcount.py
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split

# Create a local StreamingContext with two working thread and batch interval of 5 second
spark = SparkSession.builder \
    .appName("SparkMinIODemo") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", 
            "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.301") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio1:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "bigdata") \
    .config("spark.hadoop.fs.s3a.secret.key", "bigdata123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR") # reduce spam of logging

lines = spark.readStream.format('socket').option('host', 'jupyterlab').option('port', 8765).load()

words = lines.select(explode(split(lines.value, ' ')).alias('word'))

counts = words.groupby('word').count()

query = counts.writeStream.outputMode('complete').format('console').start()

query.awaitTermination()

Overwriting networkwordcount.py


Om nu de werking te starten, open in jupyterlab een terminal.
Dit is een tool die ons toelaat om tekst te vesturen naar een locale netwerkpoort om zo een tekststroom na te bootsen.
Voer daarna het volgende commando uit:
```console
    nc -lk 0.0.0.0 19999
```

Merk eerst op dat de code voor de streaming applicatie in een aparte file geplaatst is.
Ik heb dit gedaan om de volgende redenen:

* De code moet gestopt kunnen worden want dit soort applicaties blijft draaien tot er een shutdown commando komt
* Als je dit doet in een jupyterlab code-cell, dan ga je de kernel moet herstarten en terug meerdere cellen uitvoeren

Het wordcount programma kan nu gestart worden door het volgende commando uit te voeren

In [12]:
!python3 networkwordcount.py

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f48b3db2-00e4-44e1-a7e3-d4e9279a603f;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.301 in central
:: resolution report :: resolve 214ms :: artifacts dl 7ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.301 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundl

Hieronder staat een diagram om een overzicht te krijgen wat er in bovenstaande demo gebeurt:
````
                     ┌──────────────────────────┐
                     │        Your Browser       │
                     │   http://localhost:8888   │
                     └─────────────┬────────────┘
                                   │
                                   │ (port 8888)
                                   ▼
┌────────────────────────────────────────────────────────────┐
│                    Docker Network: spark-net               │
│                                                            │
│   ┌──────────────────────────┐                             │
│   │        jupyterlab         │                            │
│   │──────────────────────────│                             │
│   │ - Jupyter Notebook       │                             │
│   │ - Spark DRIVER           │                             │
│   │ - Socket Server (nc)     │◀── students type text      |
│   │   listening on :8765     │                             │
│   │                          │                             │
│   │ host: jupyterlab         │                             │
│   └─────────────┬────────────┘                             │
│                 │                                          │
│                 │ TCP socket (8765)                        │
│                 ▼                                          │
│   ┌──────────────────────────┐                             │
│   │      spark-worker-1      │                             │
│   │──────────────────────────│                             │
│   │ - Spark EXECUTOR         │                             │
│   │ - Connects to            │                             │
│   │   jupyterlab:19999       │                             │
│   └──────────────────────────┘                             │
│                                                            │
│   ┌──────────────────────────┐                             │
│   │      spark-worker-2      │                             │
│   │──────────────────────────│                             │
│   │ - Spark EXECUTOR         │                             │
│   │ - Connects to            │                             │
│   │   jupyterlab:19999       │                             │
│   └──────────────────────────┘                             │
│                                                            │
│   ┌──────────────────────────┐                             │
│   │      spark-worker-3      │                             │
│   │──────────────────────────│                             │
│   │ - Spark EXECUTOR         │                             │
│   │ - Connects to            │                             │
│   │   jupyterlab:19999       │                             │
│   └──────────────────────────┘                             │
│                                                            │
│   ┌──────────────────────────┐                             │
│   │       spark-master       │                             │
│   │──────────────────────────│                             │
│   │ - Cluster manager        │                             │
│   │ - Schedules executors    │                             │
│   └──────────────────────────┘                             │
│                                                            │
└────────────────────────────────────────────────────────────┘
````

## StreamingContext in plaats van SparkContext

Net zoals de context moet er een streaming context aangemaakt worden voor je de streaming api kan gebruiken.
Een aantal belangrijke punten om te onthouden zijn:
* Eens de context gestart is kan er geen nieuwe code toegevoegd worden
* Eens de context gestopt is kan de context niet opnieuw gestart worden. 
* Er kan maar 1 context tegelijkertijd actief zijn
* De spark context kan hergebruikt worden zolang de streaming context gestopt is voor er een nieuwe streamingcontext aangemaakt wordt.

## DStreams of Discretized Streams

**Dit is een verouderde techniek maar voor volledigheid laat ik de uitleg ervan hier staan**
De basis abstractielaag gebruikt en voorzien door Spark Streaming.
Dit stelt een continue datastroom voor die afkomstig is van de inputbron of de verwerkte datastroom van de transform stap.
Deze stream stelt een continue tijdreeks voor van RDD's.
Elk van deze RDD's stelt de ontvangen data tijdens een interval voor.

Deze steams kunnen van verscheidene bronnen komen. 
Indien je een niet-standaard geincludeerde bron wil gebruiken moet je een eigen **receiver** schrijven om de data van de bron op te halen.
Meer informatie over deze procedure vind je [hier](https://spark.apache.org/docs/latest/streaming-custom-receivers.html)

Belangrijk om te onthouden dat het aantal beschikbare cores groter moet zijn dan het aantal receivers/gebruikte databronnen.
Anders beschikt spark/de cluster niet over voldoende rekencapaciteiten om alles parallel uit te voeren.

**Beschikbare transformaties**

De meeste zaken die op een DataFrame/RDD uitgevoerd kunnen worden, kunnen ook op DStreams uitgevoerd worden. 
Een aantal operaties die extra aandacht vereisen zijn
* updateStateByKey()
* transform()
* Window operations
* Join operations

**UpdateStateByKey()**

Deze functie maakt het mogelijk om een algemene state bij te houden en up te daten bij het ontvangen van nieuwe informatie.
Voor het bovenstaande wordcount example kan dit bijvoorbeeld de wordcount van de volledige stream zijn ipv per lijn.
Pas nu het wordcount-example aan door deze twee zaken bij te houden.
Meer informatie kan je [hier](https://spark.apache.org/docs/latest/streaming-programming-guide.html#updatestatebykey-operation) vinden.

**Tip:** Het is nodig om checkpointing te configureren voor deze functie kan gebruikt worden (zodat het ergens kan bijgehouden worden). Dit gebeurt door de volgende lijn na het aanmaken van de streaming context te plaatsen:
    
    ssc.checkpoint("checkpoint")

**Transform operation**

De transform operations laat je toe om een RDD-to-RDD functie toe te passenop een DStream.
Dit laat je toe om alle RDD operaties toe te passen die niet zouden aangeboden worden door de Stream API
Het is hierbij belangrijk om op te merken dat deze functie elke batch opgeroepen wordt en dus dat het mogelijk is om parameters te wijzigen tussen de verschillende batches (aantal partities, broadcasted variabelen, ...)

    cleanedDStream = wordCounts.transform(lambda rdd: rdd.join(spamInfoRDD).filter(...))

**Window operations**

Een belangrijke eigenschap van streams is ook dat alle informatie binnen een bepaald tijdsvenster belangrijk kan zijn.
Binnen spark streams zijn er een verscheidene WindowOperations om data binnen een bepaald window te aggregeren en te verwerken.
Hieronder staat een voorbeeld om een reduce toe te passen om de 10 seconden op data dat in de laatste 30 seconden is binnengekomen.

    windowedWordCounts = pairs.reduceByKeyAndWindow(lambda x, y: x + y, lambda x, y: x - y, 30, 10)

**Join operations**

Twee streams kunnen gecombineerd worden door middel van de .join() functie.
Dit doet standaard een inner join maar andere mogelijkheden kunnen ook gekozen worden.

## Sending data to external systems

De foreachRDD functie is een krachtige functie dat toegepast wordt op elke RDD dat aangemaakt wordt door een DStream. 
Dit maakt het mogelijk om de data uit te sturen naar externe systemen en zorgt dus voor de Load-stap binnen ETL.
Een simplistische oplossing is als volgt:

    def sendRecord(rdd):
        connection = createNewConnection()  # executed at the driver
        rdd.foreach(lambda record: connection.send(record))
        connection.close()

    dstream.foreachRDD(sendRecord)
    
Bovenstaande gaat niet werken omdat de connectie door de driver aangemaakt wordt.
Deze connectie gaat geserializeerd worden en doorgestuurd naar de nodes maar dit gaat zelden correct lukken.
Connectieproblemen bij het verzenden van data komen bijna steeds voort uit het correct aanmaken op de juiste nodes van de connectie.
Een oplossing hiervoor is het volgende:

    def sendRecord(record):
        connection = createNewConnection()
        connection.send(record)
        connection.close()

    dstream.foreachRDD(lambda rdd: rdd.foreach(sendRecord))
    
Dit gaat correct werken maar is echter suboptimaal omdat in deze code, een nieuwe connectie aangemaakt wordt voor elke rij in de stream wat voor heel veel overhead zorgt.
Een andere mogelijkheid is als volgt

    def sendPartition(iter):
        connection = createNewConnection()
        for record in iter:
            connection.send(record)
        connection.close()

    dstream.foreachRDD(lambda rdd: rdd.foreachPartition(sendPartition))
    
Dit is reeds beter omdat de connectie reeds gedeeltelijk hergebruikt wordt maar wordt nog steeds herhaadelijk geopend en gesloten.
Dit zorgt nog steeds voor onnodige overhead.
De beste oplossing is door gebruik te maken van een connectionPool() waaruit connectie kunnen hergebruikt worden.
Deze pool maakt automatisch connecties uit en sluit de bestaande connecties enkel indien ze voldoende lang ongebruikt worden.

    def sendPartition(iter):
        # ConnectionPool is a static, lazily initialized pool of connections
        connection = ConnectionPool.getConnection()
        for record in iter:
            connection.send(record)
        # return to the pool for future reuse
        ConnectionPool.returnConnection(connection)

    dstream.foreachRDD(lambda rdd: rdd.foreachPartition(sendPartition))

dstream.foreachRDD(lambda rdd: rdd.foreachPartition(sendPartition))

## Checkpoints

Checkpoints is een manier om informatie op te slaan op de cluster om spark fout-tolerant te worden voor crashes van zowel de driver als individuele node.
Dit gebeurt door de nodige informatie op te slaan in een directory.

Checkpointing moet ge-enabled worden wanneer je
* een state wilt bijhouden
* een fout-tolerante applicatie wil

Let wel op dat het geen garantie is dat alle data behouden blijft maar het merendeel zou correct moeten opgevangen worden.

Checkpointing toevoegen aan je applicatie gebeurt door een directory mee te gevan aan de sparkContext waar de checkpoints in een fout-tolerant gedistribueerd opslagsysteem kunnen bijgehouden worden.
Dit gebeurd als volgt

    ssc.checkpoint("checkpoints") 

**Shared variables with checkpoints**

Accumulators en broadcasted variabelen worden niet opgeslagen door het checkpointing systeem in Spark. 
Dit kan opgelost worden door singleton instances te maken zodat ze kunnen geherinstantieerd worden nadat de driver restart na een failure.
Onderstaande code is een voorbeeld van hoe dit uit te voeren in een wordcount example.
Dit voorbeeld maakt gebruik van globals() wat de globale variabelen bijhoudt.
In dit voorbeeld wordt er gebruik gemaakt van een broadcasted array om een lijst mee te geven met beginletters van woorden die genegeerd worden.
Daarnaast wordt een accumulator gebruikt om het totaal aantal genegeerde woorden te tellen.

## Structured streaming

**Dit is de moderne variant die we gaan gebruiken in dit vak**

Naast het originele streaming systeem van spark gebruikmakende van DStreams, is er ook een [Structured Streaming](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html) variant.
Het grootste verschil is dat terwijl DStreams gebaseerd zijn op RDD's objecten maakt structured streaming gebruik van DataFrames.
Het networkcount dat we eerst gemaakt hadden hierboven is geschreven met de structured streaming aanpak.

Merk op dat we in dat voorbeeld de outputMode complete hebben gebruikt.
Als we echter ook de wordcount per batch willen weten kunnen we gebruik maken van de andere modes.
Onderstaande voorbeeld toont hoe het networkCount voorbeeld na te bootsen (namelijk via de update mode).
In totaal zijn er drie mogelijke outputmodes:

* complete: Toon alles
* append: Toon enkel de nieuw toegevoegd rijen
* update: Toon de gewijzigde kolommen

Het is belangrijk om de gebruikt outputmode in de gaten te houden want niet elke functie kan gebruikt worden met elke outputmode.

In [ ]:
%%file structuredNetworkCount.py

from pyspark.sql import SparkSession
from pyspark.sql.functions import explode
from pyspark.sql.functions import split

# Create a local StreamingContext with two working thread and batch interval of 5 second
spark = SparkSession.builder \
    .appName("SparkMinIODemo") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", 
            "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.301") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio1:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "bigdata") \
    .config("spark.hadoop.fs.s3a.secret.key", "bigdata123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR") # reduce spam of logging

In [ ]:
!python3 structuredNetworkCount.py

## Triggers

Triggers bepalen hoe vaak de applicatie binnenkomende data (micro-batches) verwerkt. Binnen pyspark zijn de volgende triggers mogelijk:

* Default: zo snel als mogelijk
* Fixed interval: vast interval om data te verwerken
* Once: voer maar 1 batch uit en stop hierna

In [ ]:
#query=counts.writeStream.outputMode('complete').format('console').start()
query=counts.writeStream \
    .outputMode('complete') \
    .format('console') \
    .trigger(processingTime='10 seconds') \
    .start()

#alternatieve optie:
# spark.readStream.option("cleanSource", "archive") # -> verwijder alle data bij de start

## Windows en watermarking

Windows gaat over het groeperen van data in tijdsvensters. Dit houdt in dat we kijken naar data die binnen een bepaalde tijdsperiode is ontvangen. Hiervoor zijn er twee belangrijke parameters:

* De duur van een tijdsvenster: hoe lang duurt elk tijdsvenster
* De stap van een tijdsvenster: hoe vaak verspringt het tijdsvenster

Een voorbeeld hiervan is een tijdsvenster van 10 seconden dat elke 2 seconden verschuift. Dit heeft als gevolg dat data die op een bepaald tijdsmoment binnenkomt, deel kan uitmaken van verschillende tijdsvensters.

Eenmaal je begint met tijdsvensters te werken kan er echter een probleem opduiken. Wanneer is een tijdsvenster afgerond en moet het niet meer gebruikt worden voor berekeningen? In principe kan er later nog data toekomen met een timestamp in dit tijdsvenster. Het zo lang actief houden van alle data is echter niet efficient. Om deze reden wordt er ook vaak gebruik gemaakt van **watermarks**. Dit wordt gebruikt om data die te laat toekomt te markeren en deze te negeren.

Hieronder staat er een voorbeeld van hoe je zowel windows als watermarks gebruikt:

In [13]:
from pyspark.sql.functions import window, current_timestamp

with_timestamps = words.withColumn('timestamp', current_timestamp()) # current_timestamp = now()

windowed_counts = with_timestamps \
    .withWatermark('timestamp', '1 minute') \
    .groupby(window(with_timestamps.timestamp, '10 seconds'), with_timestamps.word) \
    .count()

NameError: name 'words' is not defined

### Oefening

Gebruik de informatie uit [deze link](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html) en maak de volgende oefening.

**Opdracht**

1. Maak een SparkSession en lees de data via Structured Streaming van host localhost, poort 19999.
2. Splits de binnenkomende regels op in woorden.
3. Voeg een timestamp toe aan elke rij (hint: gebruik current_timestamp() of withWatermark() als je verder wil gaan).
4. Groepeer de woorden per 5 seconden venster en tel hoe vaak elk woord voorkomt binnen dat venster.
5. Toon de resultaten live op de console.

**Extra uitdagingen**
1. Filter: Negeer stopwoorden zoals the, and, a.
2. Verwijder leestekens en speciale karakters.
3. Pas het aan zodat je telt hoeveel keer elke letter voorkomt
4. Opslag: Schrijf de resultaten per venster naar een CSV-bestand in een outputmap.

In [ ]:
%%file oefeningStructured.py

from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split, current_timestamp, window, row_number, col,desc
from pyspark.sql.window import Window as W

# 1. SparkSession aanmaken
spark = SparkSession.builder \
    .appName("SparkMinIODemo") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", 
            "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.301") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio1:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "bigdata") \
    .config("spark.hadoop.fs.s3a.secret.key", "bigdata123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR") # reduce spam of logging

# 2. Streaming bron: socket lezen
lines = spark.readStream \
    .format("socket") \
    .option("host", "jupyterlab") \
    .option("port", 19999) \
    .load()

# 3. Splits elke regel in woorden
words = lines.select(
    explode(split(lines.value, " ")).alias("word"),
    current_timestamp().alias("timestamp")  # timestamp toevoegen
)

words = words.withWatermark("timestamp", "1 minute")

# 4. Tel woorden per venster van 5 seconden
windowedCounts = words.groupBy(
    window(words.timestamp, "5 seconds"),
    words.word
).count()

windowedCounts = windowedCounts.withColumn("window_start", col("window.start")) \
            .withColumn("window_end", col("window.end")) \
            .drop("window")

# 5. Resultaten naar de console schrijven
query = windowedCounts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", "false") \
    .start()

# b) Opslag in CSV
#output_path = f"s3a://04-streaming/output"
#query_csv = windowedCounts.writeStream \
#    .outputMode("append") \
#    .format("csv") \
#    .option("path", output_path) \
#    .option("checkpointLocation", output_path + "_checkpoint") \
#    .start()

query.awaitTermination()
#query_csv.awaitTermination()

In [ ]:
!python3 oefeningStructured.py

## Sources & Sinks

Net zoals een standaard spark applicatie kan Structured Streaming lezen van een reeks databronnen:

* Socket (for learning/demo)
* Kafka
* File directories (CSV, JSON, Parquet)
* Rate source (synthetic data)

en kan het ook schrijven naar de volgende bronnen:

* Console
* Files
* Kafka
* Memory (for debugging)

Een voorbeeld van hoe je kan werken met andere sources wordt op het einde uitgewerkt.
 
**Checkpointing**

Voor fouttolerantie (herstellen van errors) moet een checkpoint directory gedefinieerd worden in spark. De data in deze directory kan dan gebruikt worden om de applicatie te herstarten van de laatst opgeslagen checkpoint. Dit kan ingesteld worden bij het schrijven als volgt:

## Gebruik van andere sources en sinks

Hieronder gaan we een voorbeeld uitwerken waarbij de streaming applicatie een locatie in de MinIO cluster gaat observeren en nieuwe bestanden verwerken.

We gebruiken het bestand book.txt als databron.
De bedoeling is om:

* De file stapsgewijs naar MinIO te schrijven (alsof er “streaming” data binnenkomt). Dit wordt ook een producer van data genoemd
* Met Structured Streaming van Spark de data real-time te lezen.
* Het aantal unieke woorden te tellen.
* De resultaten in de console uit te printen.

**Producer**

Maak een file producer.py die de file in book.txt uitleest en lijn per lijn wegschrijft naar MinIO. Elke lijn wordt in een aparte json geplaatst met de volgende twee waarden:

* data: de tekst op de string
* timestamp: de huidige timestamp waarop deze lijn doorgestuurd wordt.

Om de cluster niet te overbelasten gaan we een delay toevoegen tussen twee opeenvolgende lijnen. De delay is een random verdeelde waarde tussen 100 en 1000 milliseconden.

In [ ]:
%%file producer.py
from minio import Minio
import time
from datetime import datetime
import json
import random
from io import BytesIO

client = Minio(
    "minio1:9000",
    access_key="bigdata",
    secret_key="bigdata123",
    secure=False
)

bucket = "04-streaming"
if not client.bucket_exists(bucket):
    client.make_bucket(bucket)

with open("book.txt", "r") as f:
    lines = f.readlines()

for idx, line in enumerate(lines):
    line = line.strip()
    if not line:
        continue

    # JSON bericht maken
    message = {
        "data": line,
        "timestamp": datetime.utcnow().isoformat()
    }

    # Bytes maken
    data = json.dumps(message).encode("utf-8")
    object_name = f"input/line_{idx}.json"

    # Upload naar MinIO
    client.put_object(
        bucket,
        object_name,
        data=BytesIO(data),
        length=len(data),
        content_type="application/json"
    )

    print(f"Verstuurd: {object_name} -> {message}")

    # Random delay tussen 100 en 1000 ms
    time.sleep(random.uniform(0.1, 1.0))

**Consumer**

Schrijf hieronder de nodige code voor een consumer uit te starten die de nodige stappen hierboven uitvoert.
Let er hierbij op dat er gebruik gemaakt wordt van structured streaming dat data uileest uit de bucket op MinIO

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StringType, TimestampType
from pyspark.sql.functions import explode, split, col, approx_count_distinct

spark = SparkSession.builder \
    .appName("StructuredStreamingMinIO") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio1:9000") \
    .config("spark.jars.packages", 
            "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.301") \
    .config("spark.hadoop.fs.s3a.access.key", "bigdata") \
    .config("spark.hadoop.fs.s3a.secret.key", "bigdata123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .getOrCreate()

# Streaming input directory op MinIO
input_dir = "s3a://04-streaming/input/"

## Oefeningen

Pas de consumer aan voor de volgende extra uitdagingen:
* Tel het aantal unieke woorden per tijdsvenster (14 seconden breed en laat het per 3 seconden verspringen en)
* Negeer data die na 30 seconden pas toekomt
* Zorg ervoor dat de source leeg gemaakt wordt bij het starten van de consumer.
* Complexere verwerking:
  * Stopwoorden filteren: verwijder woorden zoals "the", "and", "is".
  * Case-insensitive: maak de woorden allemaal lowercase en vergelijk.
  * Regex filtering: selecteer alleen woorden die met een hoofdletter beginnen of alleen numerieke waarden.

Meer informatie over deze uitdagingen vind je onder andere op [deze link](https://spark.apache.org/docs/latest/sql-data-sources-json.html#data-source-option)